# Simple CNN

In [3]:
import torch
import torch.nn.functional as F
from torch import nn

In [4]:
import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd

In [ ]:
# input -> conv -> relu -> pool -> conv -> relu -> pool -> fc -> softmax (output)
class CNN(nn.Module):
    """
    Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict.
           
    """
    def __init__(self, input_dim: int, output_dim: int):
        super(CNN, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(input_dim, 16, kernel_size=3, stride=1, padding=1), # Conv 1
            nn.GELU(),

            nn.AvgPool2d(kernel_size=2, stride=2), # Max polling

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # Conv 2
            nn.GELU(),
            nn.Dropout(0.2),

            nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1), # Conv 3
            nn.GELU(),
            nn.Dropout(0.3),

            nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1), # Conv 4 

        )
        self.fc = nn.Linear(64 * 3 * 3, output_dim) # Output layer, output_dim = number of classes output

    def forward(self, x):
        """
            Model flow: input -> conv -> relu -> pool -> conv -> relu -> pool -> fc -> softmax (output)
        """
        x = self.model(x)
        x = x.reshape(x.shape[0], -1)  # Flatten the tensor
        x = self.fc(x)            # Apply fully connected layer => Output layer
        return x

## Data Loading

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [7]:
BATCH_SIZE = 64
train_dataset = datasets.FashionMNIST(root='dataset/', train=True, transform=transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Lambda(lambda t: t * 2 - 1), # this is normalising to [-1, 1], not a z normalisation
                transforms.RandomHorizontalFlip(),
                # transforms.RandomVerticalFlip(),
                # transforms.GaussianBlur(3, 0.03),
            ]
        ), download=True)
test_dataset = datasets.FashionMNIST(root='dataset/', train=False, transform=transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Lambda(lambda t: t * 2 - 1), # this is normalising to [-1, 1], not a z normalisation
            ]
        ), download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [8]:
train_dataset[0][0].shape

torch.Size([1, 28, 28])

In [9]:
train_dataset.targets.unique().numel()

10

## Training

In [10]:
input_dim = train_dataset[0][0].shape[0]
output_dim = train_dataset.targets.unique().numel()

In [11]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("runs/cnn_experiment")

2026-07-02 17:30:32.670193: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-02 17:30:32.733750: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-02 17:30:34.106155: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-02 17:30:34.106531: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31]

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = CNN(input_dim=input_dim, output_dim=output_dim).to(device)

cuda


In [13]:
criterion = nn.CrossEntropyLoss() # Apply softmax as an activation func for the output layer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [14]:
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

model.0.weight 	 torch.Size([16, 1, 3, 3])
model.0.bias 	 torch.Size([16])
model.3.weight 	 torch.Size([32, 16, 3, 3])
model.3.bias 	 torch.Size([32])
model.7.weight 	 torch.Size([64, 32, 3, 3])
model.7.bias 	 torch.Size([64])
model.11.weight 	 torch.Size([64, 64, 3, 3])
model.11.bias 	 torch.Size([64])
fc.weight 	 torch.Size([10, 576])
fc.bias 	 torch.Size([10])


In [15]:
model.eval

<bound method Module.eval of CNN(
  (model): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): GELU(approximate='none')
    (5): Dropout(p=0.2, inplace=False)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): GELU(approximate='none')
    (9): Dropout(p=0.3, inplace=False)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (fc): Linear(in_features=576, out_features=10, bias=True)
)>

### Accuracy

In [16]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

    model.train()

    return num_correct / num_samples

### Running

In [17]:
epochs = 20
step = 0

for epoch in range(epochs):
    running_loss = 0.0

    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device)
        targets = targets.to(device)

        data = data + torch.randn(data.shape, device='cuda') * 0.03

        # Forward propagation
        scores = model(data)
        loss = criterion(scores, targets)

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Log batch loss
        writer.add_scalar("Training Loss / Batch", loss.item(), step)

        step += 1

    # Average loss for this epoch
    avg_epoch_loss = running_loss / len(train_loader)

    # Compute accuracy after each epoch
    train_acc = check_accuracy(train_loader, model)
    test_acc = check_accuracy(test_loader, model)

    # Log epoch metrics
    writer.add_scalar("Training Loss / Epoch", avg_epoch_loss, epoch)
    writer.add_scalar("Training Accuracy", train_acc, epoch)
    writer.add_scalar("Test Accuracy", test_acc, epoch)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {avg_epoch_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}, "
        f"Test Acc: {test_acc:.4f}"
    )

writer.close()

Epoch [1/20], Loss: 0.6060, Train Acc: 0.8464, Test Acc: 0.8377
Epoch [2/20], Loss: 0.4113, Train Acc: 0.8710, Test Acc: 0.8595
Epoch [3/20], Loss: 0.3680, Train Acc: 0.8752, Test Acc: 0.8593
Epoch [4/20], Loss: 0.3446, Train Acc: 0.8843, Test Acc: 0.8696
Epoch [5/20], Loss: 0.3296, Train Acc: 0.8920, Test Acc: 0.8735
Epoch [6/20], Loss: 0.3151, Train Acc: 0.8868, Test Acc: 0.8716
Epoch [7/20], Loss: 0.3100, Train Acc: 0.8980, Test Acc: 0.8833
Epoch [8/20], Loss: 0.3023, Train Acc: 0.9061, Test Acc: 0.8894
Epoch [9/20], Loss: 0.2946, Train Acc: 0.8940, Test Acc: 0.8757
Epoch [10/20], Loss: 0.2893, Train Acc: 0.9050, Test Acc: 0.8890
Epoch [11/20], Loss: 0.2807, Train Acc: 0.9107, Test Acc: 0.8943
Epoch [12/20], Loss: 0.2790, Train Acc: 0.9095, Test Acc: 0.8881
Epoch [13/20], Loss: 0.2743, Train Acc: 0.9086, Test Acc: 0.8918
Epoch [14/20], Loss: 0.2701, Train Acc: 0.9139, Test Acc: 0.8930
Epoch [15/20], Loss: 0.2675, Train Acc: 0.9179, Test Acc: 0.8972
Epoch [16/20], Loss: 0.2664, Train

Expected:

Fashion

* Train Acc: 93%–96%
* Test Acc:  90%–93%
* Gap:       < 3% ideally, but 4% is acceptable

Digits

* Train Acc: 97%–99%
* Test Acc:  97%–99%
* Gap:       < 2%